In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook explores the pole geometry of a continuous-time
# Bessel-Thomson low-pass filter.
#
# Two parameters can be varied:
#
#       N     : filter order
#       τ0    : group delay at ω = 0
#
# The N stable poles of H(s) are obtained from
#
#       scipy.signal.besselap(N, norm='delay')
#
# which corresponds to the normalization
#
#       τ(0) = 1.
#
# For a desired zero-frequency group delay τ0:
#
#       pk(τ0) = pk(1) / τ0
#
# The N right-half-plane poles associated with H(-s) are obtained through
#
#       pk,rejected = -pk,used
#
# Therefore H(s)H(-s) contains 2N poles.
#
# Pole representation:
#
#       Filled red circles : stable poles with Re{p} < 0 used in H(s)
#       Open red circles   : unstable poles with Re{p} > 0 rejected from H(s)
#
# A best-fit circumference is calculated for the stable Bessel poles.
# Unlike the Butterworth case, this circumference is obtained numerically
# from the actual Bessel pole positions.
#
# The information panel displays its center, radius and maximum radial error.
#
# For odd N:
#
#       one stable pole is real and negative.
#
# For even N:
#
#       all stable poles occur in complex-conjugate pairs.
#
# This notebook displays poles only; Bessel low-pass filters have no finite
# transmission zeros.
# ==============================================================================

# ==============================================================================
# DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 8px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:1080px;
    max-width:1080px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Visualize the pole geometry of a continuous-time Bessel-Thomson low-pass filter, including both the N stable poles of H(s) and their N right-half-plane counterparts associated with H(-s).
<br>
<b>Interpretation:</b>
Filled red circles denote the stable poles with Re{p}&lt;0 that are used in H(s), whereas open red circles denote the unstable right-half-plane poles with Re{p}&gt;0 that are rejected. The Bessel poles form a characteristic near-circular pattern whose geometry depends on N. A best-fit circumference is shown together with its center. Changing τ₀ scales the complete pole pattern without changing its normalized shape.
</div>
""", layout=Layout(width='1090px', max_width='1090px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='280px')
style_opts = {'description_width':'90px'}

order_slider = IntSlider(min=2, max=6, step=1, value=5, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)

tau_slider = FloatSlider(min=0.4, max=1.5, step=0.1, value=1.0, description='τ₀:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='380px', max_width='380px'))

pole_table = HTML(layout=Layout(width='460px', max_width='460px'))

# ==============================================================================
# FIGURE
# ==============================================================================

fig, ax = plt.subplots(figsize=(6.7, 6.7))

theta_circle = np.linspace(0.0, 2.0 * np.pi, 1000)

circle_line, = ax.plot([], [], 'r-', linewidth=1.2, alpha=0.65, label='Best-fit circumference')

used_scatter = ax.scatter([], [], s=95, marker='o', facecolors='red', edgecolors='red', linewidths=1.5, label='Stable / used poles')

rejected_scatter = ax.scatter([], [], s=95, marker='o', facecolors='white', edgecolors='red', linewidths=1.8, label='Unstable / rejected poles')

center_scatter = ax.scatter([], [], s=55, marker='+', color='black', linewidths=1.5, label='Circle center')

ax.axhline(0.0, color='black', linewidth=0.9)
ax.axvline(0.0, color='black', linewidth=0.9)

ax.set_xlabel('Re{s}', fontsize=11)
ax.set_ylabel('Im{s}', fontsize=11)

ax.set_title('Bessel-Thomson Pole Geometry', fontsize=13, fontweight='bold', pad=8)

ax.grid(True, linestyle=':', alpha=0.35)

ax.set_aspect('equal', adjustable='box')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=4, fontsize=8)

axis_limit = 24.0

ax.set_xlim(-axis_limit, axis_limit)
ax.set_ylim(-axis_limit, axis_limit)

ax.set_xticks(np.arange(-20, 21, 5))
ax.set_yticks(np.arange(-20, 21, 5))

fig.subplots_adjust(left=0.12, right=0.96, bottom=0.17, top=0.92)

fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.resizable = False

fig.canvas.layout.width = '670px'
fig.canvas.layout.height = '670px'

# ==============================================================================
# CIRCLE FIT
# ==============================================================================

def fit_circle(points):

    x = np.real(points)
    y = np.imag(points)

    A = np.column_stack((2.0 * x, 2.0 * y, np.ones_like(x)))

    b = x**2 + y**2

    cx, cy, c = np.linalg.lstsq(A, b, rcond=None)[0]

    radius = np.sqrt(c + cx**2 + cy**2)

    distances = np.sqrt((x - cx)**2 + (y - cy)**2)

    errors = np.abs(distances - radius)

    return cx, cy, radius, np.max(errors)

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_bessel_poles(change=None):

    N = order_slider.value
    tau0 = tau_slider.value

    # --------------------------------------------------------------------------
    # Stable Bessel poles for unit group delay
    # --------------------------------------------------------------------------

    zeros, poles_unit, gain = signal.besselap(N, norm='delay')

    # --------------------------------------------------------------------------
    # Scaling for τ(0) = τ0
    # --------------------------------------------------------------------------

    used_poles = poles_unit / tau0

    # --------------------------------------------------------------------------
    # Remaining poles of H(s)H(-s)
    # --------------------------------------------------------------------------

    rejected_poles = -used_poles

    all_poles = np.concatenate((used_poles, rejected_poles))

    # --------------------------------------------------------------------------
    # Best-fit circumference of stable poles
    # --------------------------------------------------------------------------

    center_x, center_y, radius, max_error = fit_circle(used_poles)

    circle_x = center_x + radius * np.cos(theta_circle)
    circle_y = center_y + radius * np.sin(theta_circle)

    circle_line.set_data(circle_x, circle_y)

    center_scatter.set_offsets(np.array([[center_x, center_y]]))

    # --------------------------------------------------------------------------
    # Update pole positions
    # --------------------------------------------------------------------------

    used_offsets = np.column_stack((np.real(used_poles), np.imag(used_poles)))

    rejected_offsets = np.column_stack((np.real(rejected_poles), np.imag(rejected_poles)))

    used_scatter.set_offsets(used_offsets)

    rejected_scatter.set_offsets(rejected_offsets)

    # --------------------------------------------------------------------------
    # Pole table
    # --------------------------------------------------------------------------

    rows = ""

    for index, p in enumerate(all_poles):

        if index < N:

            status = "STABLE"

            status_style = """
                color:#0066cc;
                background:#eef6ff;
                border:1px solid #9bc8f5;
            """

        else:

            status = "UNSTABLE"

            status_style = """
                color:#cc0000;
                background:#fff1f1;
                border:1px solid #efaaaa;
            """

        rows += f"""
        <tr style="border-bottom:1px solid #eeeeee;">

            <td style="
                padding:5px 8px;
                text-align:center;
                font-family:'Times New Roman',serif;
                font-size:17px;
                font-style:italic;
                white-space:nowrap;
            ">
                p<sub>{index}</sub>
            </td>

            <td style="
                padding:5px 10px;
                font-family:'Times New Roman',serif;
                font-size:16px;
                white-space:nowrap;
            ">
                {p.real:+.6f}
                <span style="font-style:italic;">{p.imag:+.6f}j</span>
            </td>

            <td style="
                padding:5px 8px;
                text-align:center;
            ">
                <span style="
                    {status_style}
                    padding:2px 7px;
                    border-radius:10px;
                    font-size:10px;
                    font-weight:bold;
                    letter-spacing:0.3px;
                    white-space:nowrap;
                ">
                    {status}
                </span>
            </td>

        </tr>
        """

    pole_table.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px;
        background:white;
        width:450px;
        max-height:630px;
        overflow-y:auto;
        box-sizing:border-box;
        font-size:12px;
    ">

    <div style="
        font-family:'Times New Roman',serif;
        font-size:18px;
        font-weight:bold;
        margin-bottom:7px;
        text-align:center;
    ">
        Pole Values
    </div>

    <table style="
        width:100%;
        border-collapse:collapse;
    ">

        <tr style="border-bottom:1px solid #bbbbbb;">
            <th style="padding:5px;">Pole</th>
            <th style="padding:5px;">Complex Value</th>
            <th style="padding:5px;">Status</th>
        </tr>

        {rows}

    </table>

    </div>
    """

    # --------------------------------------------------------------------------
    # Order information
    # --------------------------------------------------------------------------

    if N % 2 == 1:

        parity_text = "Odd order"

        real_candidates = used_poles[np.abs(np.imag(used_poles)) < 1e-10]

        if len(real_candidates) > 0:
            real_pole_text = f"One stable real pole at s = {real_candidates[0].real:.4f}"
        else:
            real_pole_text = "One stable real pole"

    else:

        parity_text = "Even order"
        real_pole_text = "No real pole"

    # --------------------------------------------------------------------------
    # Radial information
    # --------------------------------------------------------------------------

    pole_radii = np.abs(used_poles)

    min_radius = np.min(pole_radii)

    max_radius = np.max(pole_radii)

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 10px;
        margin-top:8px;
        font-size:12px;
        line-height:1.65;
        background:white;
        width:375px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Bessel-Thomson low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Group delay:</b>
        <span style="color:#0066cc;">τ₀ = {tau0:.2f} s</span>
    </div>

    <div>
        <b>Normalization:</b>
        <span style="color:#0066cc;">delay</span>
    </div>

    <div style="
        margin-top:5px;
        padding-top:5px;
        border-top:1px solid #eeeeee;
    ">
        <b>Best-fit circumference:</b>
    </div>

    <div>
        <b>Center:</b>
        <span style="color:#0066cc;">
        C = {center_x:+.6f} {center_y:+.6f}j
        </span>
    </div>

    <div>
        <b>Radius:</b>
        <span style="color:#0066cc;">R = {radius:.6f}</span>
    </div>

    <div>
        <b>Maximum radial error:</b>
        <span style="color:#0066cc;">{max_error:.6e}</span>
    </div>

    <div>
        <b>Minimum |p|:</b>
        <span style="color:#0066cc;">{min_radius:.6f}</span>
    </div>

    <div>
        <b>Maximum |p|:</b>
        <span style="color:#0066cc;">{max_radius:.6f}</span>
    </div>

    <div style="
        margin-top:5px;
        padding-top:5px;
        border-top:1px solid #eeeeee;
    ">
        <b>Total poles:</b>
        <span style="color:#0066cc;">{2 * N}</span>
    </div>

    <div>
        <b>Stable poles:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Unstable poles:</b>
        <span style="color:#cc0000;">{N}</span>
    </div>

    <div>
        <b>Order type:</b>
        <span style="color:#0066cc;">{parity_text}</span>
    </div>

    <div>
        <b>Real pole:</b>
        <span style="color:#0066cc;">{real_pole_text}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        The stable Bessel poles follow a characteristic near-circular pattern
        whose center and radius depend on N. Changing τ₀ scales the complete
        geometry by 1/τ₀. Stable left-half-plane poles are used in H(s),
        whereas unstable right-half-plane poles are rejected.
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # Redraw
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_bessel_poles, names='value')

tau_slider.observe(update_bessel_poles, names='value')

# ==============================================================================
# LAYOUT
# ==============================================================================

controls = VBox([parameter_title, order_slider, tau_slider, info_html], layout=Layout(width='390px', min_width='390px', max_width='390px', flex='0 0 390px', align_items='flex-start'))

main_row = HBox([controls, fig.canvas, pole_table], layout=Layout(width='1530px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_bessel_poles()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_row)